In [1]:
# Cargar el CVS
import pandas as pd
import numpy as np

consumo_granada = pd.read_csv('../data/processed/consumo_granada_cleaned.csv')

# Creamos una copia explícita para trabajar las features
df_features = consumo_granada.copy()

# Aseguramos formato fecha
df_features['timestamp'] = pd.to_datetime(df_features['timestamp'])

In [2]:
# 2. Extracción de componentes temporales
# Extraemos información básica que el modelo necesita entender
df_features['hour'] = df_features['timestamp'].dt.hour
df_features['month'] = df_features['timestamp'].dt.month
df_features['day_of_week'] = df_features['timestamp'].dt.dayofweek  # 0=Lunes, 6=Domingo
df_features['day_of_month'] = df_features['timestamp'].dt.day
df_features['year'] = df_features['timestamp'].dt.year  # Año para dashboard

print("Variables horarias básicas extraídas.")

Variables horarias básicas extraídas.


In [3]:
# 3. Transformación Cíclica del Tiempo (Seno y Coseno)
# Esto permite al modelo entender la continuidad entre las 23:00 y las 00:00

# Ciclo Diario (24 horas)
df_features['hour_sin'] = np.sin(2 * np.pi * df_features['hour'] / 24)
df_features['hour_cos'] = np.cos(2 * np.pi * df_features['hour'] / 24)

# Ciclo Anual (12 meses) - Opcional pero recomendado para estacionalidad
df_features['month_sin'] = np.sin(2 * np.pi * df_features['month'] / 12)
df_features['month_cos'] = np.cos(2 * np.pi * df_features['month'] / 12)

# Visualizamos para asegurar que los valores están entre -1 y 1
df_features[['hour', 'hour_sin', 'hour_cos']].head(3)

,hour,hour_sin,hour_cos
0,0,0.0,1.0
1,0,0.0,1.0
2,0,0.0,1.0


In [ ]:
# 4. Variables de Calendario (Sociales)
# A. Fin de Semana (Variable Crítica según EDA)
# Si es Sábado (5) o Domingo (6) -> 1, resto -> 0
df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)

# B. Festivos (Lógica simplificada para Granada)
# Fechas fijas nacionales/autonómicas clave
festivos_fijos = [
    (1, 1),   # Año Nuevo
    (6, 1),   # Reyes
    (28, 2),  # Día de Andalucía
    (1, 5),   # Día del Trabajo
    (15, 8),  # Asunción
    (12, 10), # Hispanidad
    (1, 11),  # Todos los Santos
    (6, 12),  # Constitución
    (8, 12),  # Inmaculada
    (25, 12)  # Navidad
]

# Función para detectar festivo
def es_festivo(row):
    m, d = row.month, row.day
    # 1. Chequear festivos fijos
    if (m, d) in festivos_fijos:
        return 1
    # 2. Día de la Cruz (3 de Mayo en Granada) - Muy importante localmente
    if m == 5 and d == 3:
        return 1
    # 3. Virgen de las Angustias (Último domingo septiembre - aprox 15-30 sept)
    # (Simplificación: marcamos el 15 de sep como patrona para el modelo)
    if m == 9 and d == 15:
        return 1
    return 0

# Aplicamos la función (tarda unos segundos)
df_features['is_holiday'] = df_features['timestamp'].apply(lambda x: es_festivo(x))

# Si es festivo entre semana, actúa como fin de semana
# Creamos una variable combinada "día_no_laborable"
df_features['is_non_working'] = ((df_features['is_weekend'] == 1) | (df_features['is_holiday'] == 1)).astype(int)

print("Calendario laboral generado.")

Calendario laboral generado.


In [ ]:
# 5. Transformaciones Climáticas (Física)
# Elevamos al cuadrado para capturar la no-linealidad (Calefacción vs A/C)

df_features['temp_sq'] = df_features['temperature'] ** 2

print("Features climáticas creadas.")

Features climáticas creadas.


In [6]:
# 6. One-Hot Encoding de Zonas (Geografía)
# Usamos dtype='int8' para ahorrar memoria (ocupa 8 veces menos)

dummies_zona = pd.get_dummies(df_features['zone_name'], prefix='zona', dtype='int8')

# Unimos las nuevas columnas al dataset principal
df_features = pd.concat([df_features, dummies_zona], axis=1)

print(f"Zonas codificadas. Nuevas columnas añadidas: {dummies_zona.shape[1]}")

Zonas codificadas. Nuevas columnas añadidas: 20


In [ ]:
# 7. Limpieza Final y Optimización de Decimales

# A. Borrado de columnas que sobran (lo que ya tenías)
columnas_a_borrar = [
    'timestamp', 'zone_id', 'zone_name', 'fecha', 'hora'
]
df_final = df_features.drop(columns=columnas_a_borrar, errors='ignore')

# B. ESTRATEGIA DE REDONDEO SELECTIVO
# Definimos las columnas "intocables" (Originales de sensor)
cols_protegidas = ['consumption_kwh', 'temperature']

# Obtenemos la lista de columnas que SÍ queremos redondear
# (Son todas las columnas del DF menos las protegidas)
cols_a_redondear = [col for col in df_final.columns if col not in cols_protegidas]

# Aplicamos el redondeo SOLO a esas columnas
df_final[cols_a_redondear] = df_final[cols_a_redondear].round(4)

# Muestra para verificar:
df_final.head()

,consumption_kwh,temperature,hour,month,day_of_week,day_of_month,year,hour_sin,hour_cos,month_sin,...,zona_Mercagranada,zona_Norte_Almanjayar,zona_Pedro_Antonio,zona_Periodistas,zona_Plaza_Toros,zona_Pts_Tecnologico,zona_Realejo,zona_Sacromonte,zona_Zaidin_Nuevo,zona_Zaidin_Vergeles
0,359.454,-0.09,0,1,3,1,2015,0.0,1.0,0.5,...,0,0,0,0,0,0,0,0,0,0
1,439.901,-0.52,0,1,3,1,2015,0.0,1.0,0.5,...,0,0,0,0,0,0,0,0,0,0
2,196.527,-0.13,0,1,3,1,2015,0.0,1.0,0.5,...,0,0,0,0,0,0,0,0,0,0
3,436.102,-0.33,0,1,3,1,2015,0.0,1.0,0.5,...,0,0,0,0,0,0,1,0,0,0
4,213.958,-0.39,0,1,3,1,2015,0.0,1.0,0.5,...,0,0,0,0,0,0,0,1,0,0


In [8]:
# Guardar dataset listo para entrenar
df_final.to_csv('../data/processed/consumo_granada_modelo.csv', index=False)